# Extraction et calcul du RFM à partir des données brutes

Ce notebook charge le fichier unifié `RFM_unified_data.csv` (une ligne par transaction), le nettoie, et calcule les indicateurs Récence / Fréquence / Montant (RFM) par client à partir de l'historique réel des ventes.

## Import des librairies et configuration des chemins

In [1]:
from pathlib import Path  # Construire des chemins robustes relatifs au projet.
import os  # Créer les dossiers de sortie si besoin.
import pandas as pd  # Manipuler les données transactionnelles.

PROJECT_ROOT = Path('.')  # Racine du projet.
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'  # Dossier des données brutes.
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'  # Dossier des données préparées.
RAW_FILE = RAW_DIR / 'RFM_unified_data.csv'  # Fichier source unifié (une ligne = une transaction).
RFM_OUTPUT = PROCESSED_DIR / 'rfm_raw_table.csv'  # Table RFM par client (sortie de ce notebook).
os.makedirs(PROCESSED_DIR, exist_ok=True)  # S'assurer que le dossier de sortie existe.

## Charger les données et vérifier la qualité

Cette section charge les transactions, contrôle les doublons et les valeurs manquantes sur les colonnes clés.

In [2]:
df = pd.read_csv(RAW_FILE)  # Charger le fichier transactionnel unifié.
df['DateVente'] = pd.to_datetime(df['DateVente'])  # Convertir la date de vente en datetime.
print(f'Lignes chargées : {len(df)}')  # Afficher le nombre de transactions.
print(f'Clients uniques : {df["ClientID"].nunique()}')  # Afficher le nombre de clients uniques.

key_cols = ['ClientID', 'DateVente', 'MontantFinal']  # Colonnes indispensables au calcul du RFM.
print('Valeurs manquantes sur colonnes clés :')  # Titre du contrôle qualité.
print(df[key_cols].isnull().sum())  # Afficher les nulls sur les colonnes essentielles.
print(f'Doublons exacts : {df.duplicated().sum()}')  # Compter les lignes strictement dupliquées.
df = df.dropna(subset=key_cols).drop_duplicates()  # Nettoyer sur les colonnes essentielles uniquement.
print(f'Lignes après nettoyage : {len(df)}')  # Confirmer le volume après nettoyage.

Lignes chargées : 25000
Clients uniques : 4938
Valeurs manquantes sur colonnes clés :
ClientID        0
DateVente       0
MontantFinal    0
dtype: int64
Doublons exacts : 0
Lignes après nettoyage : 25000


## Calculer Récence, Fréquence, Montant par client

La date de référence est fixée au lendemain de la dernière vente observée dans le jeu de données.
- **Récence** : nombre de jours depuis le dernier achat du client.
- **Fréquence** : nombre total de transactions du client.
- **Montant** : somme des montants finaux (`MontantFinal`) de toutes ses transactions.

In [3]:
reference_date = df['DateVente'].max() + pd.Timedelta(days=1)  # Définir la date de référence (lendemain de la dernière vente).
print(f'Date de référence utilisée pour la Récence : {reference_date.date()}')  # Afficher la date de référence.

rfm = df.groupby('ClientID').agg(
    Recency=('DateVente', lambda x: (reference_date - x.max()).days),
    Frequency=('TransactionID', 'count'),
    Monetary=('MontantFinal', 'sum')
).reset_index()  # Agréger les transactions par client pour obtenir R, F, M.

print(f'Table RFM construite pour {len(rfm)} clients.')  # Confirmer le nombre de clients dans la table RFM.
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())  # Afficher les statistiques descriptives.

Date de référence utilisée pour la Récence : 2026-01-01


Table RFM construite pour 4938 clients.
           Recency    Frequency       Monetary
count  4938.000000  4938.000000    4938.000000
mean    243.809235     5.062778   51017.471156
std     224.959459     2.368714   57861.122955
min       1.000000     1.000000     799.500000
25%      76.000000     3.000000   15313.360750
50%     177.000000     5.000000   31117.404500
75%     347.000000     7.000000   62987.850750
max    1417.000000    16.000000  490681.006000


## Contrôles de cohérence

Vérification qu'aucune valeur négative ou aberrante n'est présente avant export.

In [4]:
negative_count = (rfm[['Recency', 'Frequency', 'Monetary']] < 0).sum().sum()  # Compter les valeurs négatives.
if negative_count > 0:  # Avertir uniquement si des valeurs négatives existent.
    print(f'ATTENTION : {negative_count} valeurs RFM négatives détectées.')  # Signaler une anomalie.
else:
    print('Aucune valeur RFM négative détectée.')  # Confirmer la cohérence des données.
print(rfm.head())  # Aperçu des premières lignes de la table RFM.

Aucune valeur RFM négative détectée.
   ClientID  Recency  Frequency    Monetary
0  CLT00001       73          6  273493.001
1  CLT00002      203         11   46828.960
2  CLT00003      107          5    8565.360
3  CLT00004       80          3   92388.608
4  CLT00005      524          4   54662.762


## Sauvegarder la table RFM par client

In [5]:
rfm.to_csv(RFM_OUTPUT, index=False)  # Sauvegarder la table RFM calculée par client.
print(f'Table RFM sauvegardée : {RFM_OUTPUT}')  # Confirmer le chemin de sortie.

Table RFM sauvegardée : data/processed/rfm_raw_table.csv
